# T-DEED 在 SoccerNet-GAR 上的評估（CS 496）

這個 notebook 會：
1. 安裝 T-DEED（你朋友的 `TEED-match-tagger` 工具 + 官方檢查點）
2. 從 Hugging Face 下載 GAR validation 的影片片段，每類最多 50 段
3. 用 T-DEED 預測每段影片的動作，跟標準答案比對
4. 輸出每類準確率、混淆矩陣，並和位置資料的模型比較

**開始之前：** 上方選單「執行階段 → 變更執行階段類型」→ 選 **T4 GPU** → 儲存。然後從上到下逐格按 ▶ 執行。

全部跑完大約 20～30 分鐘（大部分時間在下載影片）。

## 1. 確認有 GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "沒有 GPU：請到「執行階段 → 變更執行階段類型」選 T4 GPU，然後重新執行這一格"
print("GPU OK:", torch.cuda.get_device_name(0))

## 2. 安裝 T-DEED 和程式碼

In [ ]:
%cd /content
!rm -rf TEED-match-tagger sn-gar-2027
!git clone -q https://github.com/jmoussa99/TEED-match-tagger.git
!git clone -q https://github.com/SoccerNet/sn-gar-2027.git
!git clone -q https://github.com/arturxe2/T-DEED TEED-match-tagger/private/T-DEED
!git -C TEED-match-tagger/private/T-DEED checkout -q d953b4de8a2ad941c565e1683c5b07255cbe644e
!git -C TEED-match-tagger/private/T-DEED apply /content/TEED-match-tagger/app/patches/tdeed_mps.patch
!pip -q install timm gdown
print("done")

## 3. 下載 T-DEED 檢查點（185 MB，來自 T-DEED 作者的 Google Drive）

In [ ]:
CKPT_DIR = "/content/TEED-match-tagger/private/T-DEED/checkpoints/SoccerNetBall/SoccerNetBall_challenge2"
!mkdir -p {CKPT_DIR}
!gdown -q 1uCWam8EhAXt3z6QQAfhRfzHITUcml438 -O {CKPT_DIR}/checkpoint_best.pt
!ls -lh {CKPT_DIR}

## 4. 評估腳本

這格會把評估腳本寫進 `TEED-match-tagger/scripts/evaluate_gar.py`（還沒推上 GitHub，所以直接放在這裡）。

In [ ]:
%%writefile /content/TEED-match-tagger/scripts/evaluate_gar.py
"""Evaluate the T-DEED tagger on labeled SoccerNet-GAR video clips.

Every GAR clip is 4.8 s long and its labeled action happens at ~2.43 s (the clip center).
For each clip we run the same scoring pipeline as the app (app/tdeed_runner.compute_scores),
take each class's highest score within +/- WINDOW_S seconds of the action time, and predict
the class with the highest score. Predictions are restricted to the 10 GAR classes by default
(T-DEED's extra classes DRIVE and BALL PLAYER BLOCK do not exist in GAR).

Usage (from this repo's root, after scripts/setup_tdeed.sh):
  .venv/bin/python scripts/evaluate_gar.py                       # every clip found in --videos
  .venv/bin/python scripts/evaluate_gar.py --per-class 20        # at most 20 clips per class
  .venv/bin/python scripts/evaluate_gar.py --limit 10            # quick smoke test

Outputs (outputs/gar_eval/): predictions.csv, per_class.csv, confusion.csv, metrics.json
"""
from __future__ import annotations

import argparse
import json
import random
import sys
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT / "app"))
import tdeed_runner as tr  # noqa: E402

if sys.platform != "darwin":
    # The app decodes with macOS VideoToolbox ("-hwaccel videotoolbox"), which does not exist on
    # Linux (e.g. Google Colab). Re-use the app's decoder with that flag removed.
    import subprocess as _sp
    _orig_popen = _sp.Popen

    def _popen_without_videotoolbox(cmd, *args, **kwargs):
        if isinstance(cmd, list) and "videotoolbox" in cmd:
            i = cmd.index("videotoolbox")
            cmd = cmd[: i - 1] + cmd[i + 1:]
        return _orig_popen(cmd, *args, **kwargs)

    tr.subprocess.Popen = _popen_without_videotoolbox

GAR_PROJECT = REPO_ROOT.parent / "sn-gar-2027"
ACTION_TIME_S = 2.43   # alignment.action_offset_seconds in the GAR video metadata
WINDOW_S = 1.0         # +/- 1 s, the same tolerance as the SoccerNet mAP@1 metric
OUT_DIR = REPO_ROOT / "outputs" / "gar_eval"


def load_labels(annotations: Path) -> dict[str, str]:
    data = json.load(open(annotations))["data"]
    return {s["id"]: s["labels"]["action"]["label"] for s in data}


def clip_id(path: Path, split: str) -> str:
    # clip_000123.mp4 -> valid_000123
    return f"{split}_{path.stem.split('_')[-1]}"


def predict_from_scores(scores: np.ndarray, classes: list[str]) -> tuple[str, float, dict[str, float]]:
    """scores: (N, 1 + 12) per-frame probabilities at tr.SAMPLED_FPS (column 0 = background)."""
    center = int(round(ACTION_TIME_S * tr.SAMPLED_FPS))
    half = int(round(WINDOW_S * tr.SAMPLED_FPS))
    lo, hi = max(0, center - half), min(len(scores), center + half + 1)
    window = scores[lo:hi, 1:]                      # drop background
    peak = window.max(axis=0) if len(window) else np.zeros(len(tr.CLASSES))
    per_class = {c: float(peak[i]) for i, c in enumerate(tr.CLASSES)}
    best = max(classes, key=lambda c: per_class[c])
    return best, per_class[best], per_class


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--split", default="valid", choices=["train", "valid", "test"])
    ap.add_argument("--annotations", type=Path, default=None,
                    help="GAR annotations JSON (default: ../sn-gar-2027/data/annotations_<split>.json)")
    ap.add_argument("--videos", type=Path, default=None,
                    help="folder of GAR clips (default: ../sn-gar-2027/data/videos/<split>)")
    ap.add_argument("--per-class", type=int, default=None, help="max clips per class")
    ap.add_argument("--limit", type=int, default=None, help="max clips in total")
    ap.add_argument("--all-12", action="store_true", help="allow DRIVE / BALL PLAYER BLOCK predictions")
    ap.add_argument("--device", default="auto", choices=["auto", "cuda", "mps", "cpu"],
                    help="auto = cuda if available, else mps, else cpu")
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()

    annotations = a.annotations or GAR_PROJECT / "data" / f"annotations_{a.split}.json"
    videos_dir = a.videos or GAR_PROJECT / "data" / "videos" / a.split
    labels = load_labels(annotations)

    clips = sorted(videos_dir.glob("clip_*.mp4"))
    clips = [p for p in clips if clip_id(p, a.split) in labels]
    if not clips:
        sys.exit(f"No labeled clips found in {videos_dir}")
    if a.per_class:
        rng = random.Random(a.seed)
        by_class: dict[str, list[Path]] = {}
        for p in clips:
            by_class.setdefault(labels[clip_id(p, a.split)], []).append(p)
        clips = sorted(p for ps in by_class.values() for p in rng.sample(ps, min(a.per_class, len(ps))))
    if a.limit:
        clips = clips[: a.limit]
    print(f"{len(clips)} clips:", dict(Counter(labels[clip_id(p, a.split)] for p in clips).most_common()))

    classes = tr.CLASSES if a.all_12 else tr.GAR_CLASSES
    device = a.device
    if device == "auto" and tr.torch.cuda.is_available():
        device = "cuda"
    model = tr.get_model(device)
    print(f"Model on {tr.pick_device(device)}")
    rows, t0 = [], time.perf_counter()
    for i, p in enumerate(clips, 1):
        scores = tr.compute_scores(p, model=model)
        pred, conf, per_class = predict_from_scores(scores, classes)
        cid = clip_id(p, a.split)
        rows.append({"id": cid, "label": labels[cid], "pred": pred, "confidence": round(conf, 4),
                     **{f"score_{c}": round(v, 4) for c, v in per_class.items()}})
        if i % 10 == 0 or i == len(clips):
            el = time.perf_counter() - t0
            print(f"  {i}/{len(clips)} clips · {el / i:.1f} s per clip · ETA {el / i * (len(clips) - i) / 60:.1f} min",
                  flush=True)

    df = pd.DataFrame(rows)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT_DIR / "predictions.csv", index=False)

    present = [c for c in tr.GAR_CLASSES if (df.label == c).any()]
    recall = {c: float((df[df.label == c].pred == c).mean()) for c in present}
    precision = {c: float((df[df.pred == c].label == c).mean()) if (df.pred == c).any() else float("nan")
                 for c in present}
    per_class = pd.DataFrame({"class": present,
                              "n_clips": [int((df.label == c).sum()) for c in present],
                              "recall": [round(recall[c], 3) for c in present],
                              "precision": [round(precision[c], 3) for c in present]})
    per_class.to_csv(OUT_DIR / "per_class.csv", index=False)
    pd.crosstab(df.label, df.pred).to_csv(OUT_DIR / "confusion.csv")
    metrics = {"split": a.split, "n_clips": len(df), "classes_present": len(present),
               "prediction_classes": "all 12" if a.all_12 else "GAR 10",
               "window_seconds": WINDOW_S, "action_time_seconds": ACTION_TIME_S,
               "balanced_accuracy": 100 * float(np.mean(list(recall.values()))),
               "accuracy": 100 * float((df.label == df.pred).mean()),
               "seconds_per_clip": round((time.perf_counter() - t0) / len(df), 2)}
    json.dump(metrics, open(OUT_DIR / "metrics.json", "w"), indent=2)

    print(json.dumps(metrics, indent=2))
    print(per_class.to_string(index=False))
    print(f"\nSaved to {OUT_DIR}")


if __name__ == "__main__":
    main()

## 5. 登入 Hugging Face

SoccerNet-GAR 需要存取權限。到 https://huggingface.co/settings/tokens 複製你的 **Read** token，執行這格後貼上（輸入時不會顯示）。

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass("Hugging Face token: "))

## 6. 下載影片片段

從 validation 的 12,091 段裡，每類隨機抽最多 `PER_CLASS` 段（少見類別全部都拿）。
影片分散在 35 個約 1 GB 的壓縮檔裡；這格一次下載一個，只解出需要的片段，然後立刻刪掉壓縮檔，所以不會佔用太多空間。

In [ ]:
import json, random, collections, tarfile, os
from pathlib import Path
import pandas as pd
from huggingface_hub import hf_hub_download

PER_CLASS = 50
REPO = "OpenSportsLab/SoccerNet-GAR"
OUT = Path("/content/sn-gar-2027/data/videos")
TMP = Path("/content/hf_tmp")

ann = json.load(open("/content/sn-gar-2027/data/annotations_valid.json"))["data"]
by_class = collections.defaultdict(list)
for s in ann:
    by_class[s["labels"]["action"]["label"]].append(s["id"])
rng = random.Random(0)
chosen = {i for ids in by_class.values() for i in rng.sample(sorted(ids), min(PER_CLASS, len(ids)))}
label = {s["id"]: s["labels"]["action"]["label"] for s in ann}
print(len(chosen), "clips:", dict(collections.Counter(label[i] for i in chosen).most_common()))

man = pd.read_parquet(hf_hub_download(REPO, "valid/shard_manifest.parquet", repo_type="dataset", revision="videos"))
man = man[man.sample_id.isin(chosen) & (man.status == "ok")]
shards = sorted(man.shard_name.unique())
for k, (shard, rows) in enumerate(man.groupby("shard_name"), 1):
    want = {m: OUT / p for m, p in zip(rows.wds_member, rows.relative_path) if not (OUT / p).exists()}
    if want:
        tar_path = hf_hub_download(REPO, f"valid/shards/{shard}", repo_type="dataset", revision="videos", local_dir=TMP)
        with tarfile.open(tar_path) as tf:
            for mem in tf:
                if mem.name in want:
                    want[mem.name].parent.mkdir(parents=True, exist_ok=True)
                    want[mem.name].write_bytes(tf.extractfile(mem).read())
        os.remove(tar_path)
    print(f"{k}/{len(shards)} {shard}: {len(rows)} clips", flush=True)
print("clips on disk:", len(list((OUT / "valid").glob("*.mp4"))))

## 7. 先用 5 段測試（確認一切正常，看每段要幾秒）

In [ ]:
%cd /content/TEED-match-tagger
!python scripts/evaluate_gar.py --limit 5

## 8. 完整評估（所有下載的片段）

In [ ]:
%cd /content/TEED-match-tagger
!python scripts/evaluate_gar.py

## 9. 結果和比較

和位置資料的模型比較：官方 GIN baseline（test split）和我們 Milestone 1 的特徵模型（validation）。
注意 T-DEED 是在每類最多 50 段的抽樣上評估的，所以比較是參考性質。

In [ ]:
import json, pandas as pd, numpy as np
import matplotlib.pyplot as plt
R = "/content/TEED-match-tagger/outputs/gar_eval"
m = json.load(open(f"{R}/metrics.json")); pc = pd.read_csv(f"{R}/per_class.csv")
print(json.dumps(m, indent=2))

baseline_test = {"PASS": .798, "PLAYER SUCCESSFUL TACKLE": .542, "OUT": .912, "HEADER": .655, "THROW IN": .839,
                 "CROSS": .885, "FREE KICK": .897, "SHOT": .769, "GOAL": .733, "HIGH PASS": .833}
ours_m1_valid = {"PASS": .661, "PLAYER SUCCESSFUL TACKLE": .378, "OUT": .633, "HEADER": .813, "THROW IN": .809,
                 "CROSS": .806, "FREE KICK": .761, "SHOT": .644, "GOAL": .174, "HIGH PASS": .154}
cmp = pc.rename(columns={"recall": "T-DEED (video)"})[["class", "n_clips", "T-DEED (video)"]]
cmp["GIN baseline (tracking)"] = cmp["class"].map(baseline_test)
cmp["Our M1 features (tracking)"] = cmp["class"].map(ours_m1_valid)
print("\nBalanced accuracy — T-DEED: %.1f%% | GIN baseline: 78.6%% | our M1: 58.3%%" % m["balanced_accuracy"])
display(cmp)

cm = pd.read_csv(f"{R}/confusion.csv", index_col=0)
cm = cm.reindex(index=cmp["class"], columns=cmp["class"], fill_value=0)
cmn = cm.div(cm.sum(1), axis=0)
names = [c.replace("PLAYER SUCCESSFUL ", "") for c in cmn.index]
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.imshow(cmn.values, cmap="Blues", vmin=0, vmax=1)
for i in range(len(names)):
    for j in range(len(names)):
        if cmn.values[i, j] >= .05:
            ax.text(j, i, f"{cmn.values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if cmn.values[i, j] > .55 else "black")
ax.set_xticks(range(len(names)), names, rotation=45, ha="right"); ax.set_yticks(range(len(names)), names)
ax.set_xlabel("predicted"); ax.set_ylabel("true action"); ax.set_title("T-DEED on GAR validation clips (rows sum to 1)", loc="left")
plt.tight_layout(); plt.savefig(f"{R}/confusion_matrix.png", dpi=150); plt.show()
cmp.to_csv(f"{R}/comparison.csv", index=False)

## 10. 下載結果（會下載一個 zip，傳回來給 Claude 或放進 repo）

In [ ]:
!cd /content/TEED-match-tagger/outputs && zip -qr /content/gar_eval_results.zip gar_eval
from google.colab import files
files.download("/content/gar_eval_results.zip")